# MSIRI/MCIA — Sentinel-2 Cloud Reconstruction: Baseline U-Net (Kaggle)

Trains the reference-conditioned U-Net baseline on the synthetic supervised dataset.

**Before running:** Notebook settings (right sidebar) -> **Accelerator: GPU T4 x2** (or P100/L4), **Internet: On**, and **Add Input** -> your `synthetic_dataset` Kaggle Dataset.

Run the cells top to bottom. Cell 4 (fast-dev) and Cell 5 (sanity) take ~1 minute each and must pass before you start the full run in Cell 6.

## 1. Get the code and install the one extra dependency
Edit `REPO_URL` to your GitHub repo.

In [ ]:
REPO_URL = "https://github.com/<you>/<repo>.git"   # <-- your repo
!git clone -q $REPO_URL /kaggle/working/trial3
%cd /kaggle/working/trial3
!pip install -q rasterio      # needed by the dataset import chain; torch/pandas/tb are preinstalled
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## 2. Locate (or extract) the dataset
Two options — pick ONE and set `DATA_ROOT` accordingly.

* **A (recommended):** you uploaded a single `synthetic_dataset.tar` -> extract to fast scratch `/kaggle/temp`.
* **B:** you uploaded the raw folder -> read it directly from `/kaggle/input`.

Replace `<dataset-slug>` with your dataset's slug (see the path under **Input** in the sidebar).

In [ ]:
import os, glob
SLUG = "<dataset-slug>"   # e.g. msiri-s2-synthetic

# --- Option A: single tar -> extract to /kaggle/temp (fast, not size-limited like /kaggle/working) ---
tar = glob.glob(f"/kaggle/input/{SLUG}/*.tar")
if tar:
    os.makedirs("/kaggle/temp/data", exist_ok=True)
    !tar -xf {tar[0]} -C /kaggle/temp/data
    DATA_ROOT = glob.glob("/kaggle/temp/data/**/samples", recursive=True)[0].rsplit("/", 1)[0]
else:
    # --- Option B: raw folder mounted read-only ---
    DATA_ROOT = glob.glob(f"/kaggle/input/{SLUG}/**/samples", recursive=True)[0].rsplit("/", 1)[0]

print("DATA_ROOT =", DATA_ROOT)
print("contents:", sorted(os.listdir(DATA_ROOT)))   # expect: patch_library cloud_tile_library samples
print("train samples:", len(glob.glob(f"{DATA_ROOT}/samples/train/*.json")))

## 3. Point the config at the dataset
Patches only the `data.root:` line of `configs/unet_baseline.yaml`.

In [ ]:
import re, pathlib
p = pathlib.Path("configs/unet_baseline.yaml")
p.write_text(re.sub(r"root:.*", f"root: {DATA_ROOT}", p.read_text(), count=1))
print([l for l in p.read_text().splitlines() if l.strip().startswith("root:")])

## 4. Fast-dev dry run (~30 s) — exercises the WHOLE pipeline on a few batches
Confirms data loading, model, loss, metrics, checkpointing, visualisation and the summary all work before you spend GPU hours.

In [ ]:
!python run_train.py --config configs/unet_baseline.yaml --limit-batches 3 --epochs 1 --name dryrun

## 5. Sanity check (~1 min) — proves the model actually learns
Overfits 64 samples for 2 epochs and asserts the loss drops. Prints `SANITY PASSED/FAILED`.

In [ ]:
!python run_train.py --config configs/unet_baseline.yaml --sanity

## 6. Full training
Outputs -> `/kaggle/working/experiments/unet_baseline/` (checkpoints, metrics.csv, visualizations, tensorboard, model_summary.json). `resume: auto` is set, so re-running this cell after an interruption continues from the last epoch.

In [ ]:
!python run_train.py --config configs/unet_baseline.yaml

## 7. Monitor (run in parallel / after)

In [ ]:
import pandas as pd
run = "/kaggle/working/experiments/unet_baseline"
print(open(f"{run}/model_summary.json").read())
display(pd.read_csv(f"{run}/metrics.csv").tail())
%load_ext tensorboard
%tensorboard --logdir {run}/tensorboard

## 8. Resume after a Kaggle session ends
1. **Save Version** (commit) so `/kaggle/working` is persisted as this notebook's output.
2. New session -> **Add Input -> your own notebook's previous output**, then re-run Cells 1-3 and Cell 6. `resume: auto` finds `latest.pt` (in `/kaggle/working`, or under `/kaggle/input` if re-attached) and continues.

## 9. Evaluate the best checkpoint + save the model

In [ ]:
CKPT = "/kaggle/working/experiments/unet_baseline/checkpoints/best.pt"
!python run_eval.py --checkpoint {CKPT} --split test --out /kaggle/working/eval_test.json
print(open("/kaggle/working/eval_test.json").read())
# Then 'Save Version' and download best.pt + reports from the notebook's Output tab.